In [ ]:
### DeQA, Q-Align Score Distribution ###
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

iqa_metric = "deqa_score"
base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset"
iqa_csv_file_path = os.path.join(base_image_dir, f"{iqa_metric}.csv")
df = pd.read_csv(iqa_csv_file_path)

print(f"=== {iqa_metric} 数据概览 ===")
print(f"总样本数: {len(df)}")
print(f"{iqa_metric} 统计信息:")
print(df[iqa_metric].describe())

print(f"\n{iqa_metric} 范围: {df[iqa_metric].min():.3f} - {df[iqa_metric].max():.3f}")
print(f"平均值: {df[iqa_metric].mean():.3f}")
print(f"标准差: {df[iqa_metric].std():.3f}")

score_threshold = 4.25
high_score_samples = df[df[iqa_metric] > score_threshold]
print(f"high {iqa_metric} > {score_threshold} sample num: {len(high_score_samples)}")

plt.figure(figsize=(10, 6), dpi=300)
sns.kdeplot(data=df, x=iqa_metric, fill=True, alpha=0.6)
plt.title(f'{iqa_metric}', fontsize=14, fontweight='bold')
plt.xlabel(iqa_metric, fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
### DeQA, Q-Align Score High-Quality-Image ###
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

iqa_threshold = 4.25
iqa_metric_list = ["deqa_score", "qalign_score"]
base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset"

all_information_csv_path = os.path.join(base_image_dir, "train.csv")
high_quality_image_csv_path = os.path.join(base_image_dir, "train_high_quality.csv")

# Read all metric CSV files
metric_dataframes = {}
for metric in iqa_metric_list:
    metric_csv_path = os.path.join(base_image_dir, f"{metric}.csv")
    metric_df = pd.read_csv(metric_csv_path)
    metric_dataframes[metric] = metric_df
    print(f"Loaded {metric}: {len(metric_df)} records")

# Read the main information CSV
all_info_df = pd.read_csv(all_information_csv_path)
print(f"Main information CSV: {len(all_info_df)} records")

# Merge all metric data into the main information table
merged_df = all_info_df.copy()
for metric, df in metric_dataframes.items():
    merged_df = merged_df.merge(df[['uid', metric]], on='uid', how='inner')

print(f"Data after merging: {len(merged_df)} records")

# Filter samples where both metrics are greater than the threshold
high_quality_mask = True
for metric in iqa_metric_list:
    high_quality_mask = high_quality_mask & (merged_df[metric] > iqa_threshold)

high_quality_df = merged_df[high_quality_mask].copy()

print(f"\nFilter condition: {iqa_metric_list} all > {iqa_threshold}")
print(f"Number of high-quality samples: {len(high_quality_df)}")
print(f"Percentage of total samples: {len(high_quality_df) / len(merged_df) * 100:.2f}%")

# Display statistics for high-quality samples
print(f"\nStatistics for high-quality samples:")
for metric in iqa_metric_list:
    print(f"{metric} - Mean: {high_quality_df[metric].mean():.3f}, "
          f"Range: {high_quality_df[metric].min():.3f} - {high_quality_df[metric].max():.3f}")

# Save high-quality samples to new CSV file
high_quality_df.to_csv(high_quality_image_csv_path, index=False)
print(f"\nHigh-quality sample information saved to: {high_quality_image_csv_path}")

# Display saved column information
print(f"Saved columns: {list(high_quality_df.columns)}")


In [ ]:
#### split train, val, test
import os
import pandas as pd
from sklearn.model_selection import train_test_split

base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset"
high_quality_csv_path = os.path.join(base_image_dir, "high_quality.csv")

train_path = os.path.join(base_image_dir, "high_quality_train.csv")
val_path = os.path.join(base_image_dir, "high_quality_val.csv")
test_path = os.path.join(base_image_dir, "high_quality_test.csv")

df = pd.read_csv(high_quality_csv_path)
print(f"Total high-quality samples: {len(df)}")

if df['uid'].duplicated().any():
    print("Warning: Duplicate 'uid' found. Consider deduplication.")
    df = df.drop_duplicates(subset=['uid']).reset_index(drop=True)
    print(f"After deduplication: {len(df)} samples")

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 第一步：从全集划分出 train + (val+test)
train_df, temp_df = train_test_split(
    df,
    test_size=(val_ratio + test_ratio),
    random_state=42,
    shuffle=True
)

# 第二步：将 temp_df 划分为 val 和 test（各占剩余部分的一半）
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,  # 因为 val:test = 15:15 = 1:1
    random_state=42,
    shuffle=True
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

# 保存
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("\n✅ Split completed and saved:")
print(f"  Train: {train_path}")
print(f"  Val:   {val_path}")
print(f"  Test:  {test_path}")

# 可选：验证无重叠
train_uids = set(train_df['uid'])
val_uids = set(val_df['uid'])
test_uids = set(test_df['uid'])

assert len(train_uids & val_uids) == 0, "Train and Val overlap!"
assert len(train_uids & test_uids) == 0, "Train and Test overlap!"
assert len(val_uids & test_uids) == 0, "Val and Test overlap!"
print("✅ No overlap between splits.")

In [ ]:
#### Link high-quality-image ####
import os
import pandas as pd
from tqdm import tqdm

base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset"
high_quality_image_csv_path = os.path.join(base_image_dir, "high_quality_test.csv")
high_quality_df = pd.read_csv(high_quality_image_csv_path)

base_source_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/all"
base_target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_test"
method_list = [ "real", "SD_1.4", "SD_2.1", "SDXL_1.0", "SD_3", "PA_alpha" ]
ext_list = [".png", ".PNG", ".jpg", ".jpeg", ".JPG", ".JPEG"]

# Create target directory if it doesn't exist
os.makedirs(base_target_image_dir, exist_ok=True)

# Get all unique uids from high-quality dataset
high_quality_uids = set(high_quality_df['uid'].tolist())
print(f"Total high-quality samples: {len(high_quality_uids)}")

# For each method, find and link high-quality images
linked_count = 0
for method in method_list:
    source_method_dir = os.path.join(base_source_image_dir, method)
    target_method_dir = os.path.join(base_target_image_dir, method)
    
    # Create target method directory if it doesn't exist
    os.makedirs(target_method_dir, exist_ok=True)
    
    print(f"Processing method: {method}")
    
    # Use tqdm for progress bar
    for uid in tqdm(high_quality_uids, desc=f"Processing {method}", unit="uid"):
        for ext in ext_list:
            source_file = os.path.join(source_method_dir, f"{uid}{ext}")
            target_file = os.path.join(target_method_dir, f"{uid}{ext}")
            
            # Check if source file exists and create symlink
            if os.path.exists(source_file):
                try:
                    if not os.path.exists(target_file):
                        os.symlink(source_file, target_file)
                        linked_count += 1
                        break  # Found the file, no need to check other extensions
                except FileExistsError:
                    pass  # File already exists, continue
                except Exception as e:
                    print(f"Error linking {source_file}: {str(e)}")

print(f"\nTotal files linked: {linked_count}")
print(f"High-quality images have been linked to: {base_target_image_dir}")

In [ ]:
#### construct the paired train dataset ####
import os
import pandas as pd
from tqdm import tqdm

ext_list = [ ".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG" ]
all_information_csv_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train.csv"
all_information_csv = pd.read_csv(all_information_csv_path)

generator_list = [ "SD_1.4", "SD_2.1", "SD_3" ]
base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train"
real_image_dir = os.path.join(base_image_dir, "real")
target_information_csv_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train/SD_1.4-SD_2.1_SD_3.csv"

records = []
for generator in generator_list:
    model_image_dir = os.path.join(base_image_dir, generator)
    for image_name in tqdm(os.listdir(model_image_dir), desc=generator):
        uid, _ = os.path.splitext(image_name)
        lose_image_path = os.path.join(model_image_dir, image_name)
        
        win_image_path = None
        for ext in ext_list:
            win_image_path = os.path.join(real_image_dir, f"{uid}{ext}")
            if os.path.exists(win_image_path):
                break
        
        matched_row = all_information_csv.loc[all_information_csv['uid'] == uid].iloc[0]
        prompt = matched_row["PROMPT"]
        records.append({
            "uid": uid,
            "prompt": prompt,
            "win_image_path": win_image_path,
            "lose_image_path": lose_image_path
        })
pd.DataFrame(records).to_csv(target_information_csv_path, index=True)       
print(f"records num: {len(records)}") 

In [ ]:
#### construct the add_noise-denoise csv file ####
import os
import pandas as pd
from tqdm import tqdm

ext_list = [ ".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG" ]
all_information_csv_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/high_quality_train.csv"
all_information_csv = pd.read_csv(all_information_csv_path)

generator_list = ["fake"]
base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/25"
real_image_dir = os.path.join(base_image_dir, "real")
target_information_csv_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/25/train.csv"

records = []
for generator in generator_list:
    model_image_dir = os.path.join(base_image_dir, generator)
    for image_name in tqdm(os.listdir(model_image_dir), desc=generator):
        uid, _ = os.path.splitext(image_name)
        lose_image_path = os.path.join(model_image_dir, image_name)
        
        win_image_path = None
        for ext in ext_list:
            win_image_path = os.path.join(real_image_dir, f"{uid}{ext}")
            if os.path.exists(win_image_path):
                break
        
        matched_row = all_information_csv.loc[all_information_csv['uid'] == uid].iloc[0]
        prompt = matched_row["PROMPT"]
        records.append({
            "uid": uid,
            "prompt": prompt,
            "win_image_path": win_image_path,
            "lose_image_path": lose_image_path
        })
pd.DataFrame(records).to_csv(target_information_csv_path, index=False)       
print(f"records num: {len(records)}") 

In [ ]:
#### Get Remain Image ####
import os
import pandas as pd
from tqdm import tqdm

select_uid_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/paired_real_generated_dataset/deqa_score.csv"
select_uid_df = pd.read_csv(select_uid_file_path)
select_uid_set = set(select_uid_df['uid'].tolist())

train_csv_path = "/data3/chenweiyan/dataset/Text-Complexity/data-split-20241028/all-train.csv"
train_csv = pd.read_csv(train_csv_path)

remain_mask = ~train_csv['uid'].isin(select_uid_set)
remain_uid_df = train_csv.loc[remain_mask, ['uid']]
remain_uid_df.to_csv('/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/remain_train_uid.csv', index=False)
print(f"train csv num: {len(train_csv)}")
print(f"select uid num: {len(select_uid_set)}")
print(f"remain uid num: {remain_mask.sum()}")